*Codigo del Libro*

In [1]:
from random import random, randint, sample
from collections import namedtuple

In [3]:


# Calcula el capital invertido por un individuo
def capitalInvertido(individuo):
    return sum(map(lambda x,y: x*y.precio, individuo, inversiones))

# Calcula el rendimiento obtenido por un individuo
def rendimiento(individuo):
    return sum(map(lambda x,y: x*y.precio*y.rendim, individuo, inversiones))

# Si un individuo gasta más capital del disponible, se eliminan
# aleatoriamente inversiones hasta que se ajusta al capital
def ajustaCapital(individuo):
    ajustado = individuo[:]
    while capitalInvertido(ajustado) > capital:
        pos = randint(0, len(ajustado) - 1)
        if ajustado[pos] > 0:
            ajustado[pos] -= 1
            
    return ajustado

# Crea un individuo al azar, en este caso una selección de
# inversiones que no excedan el capital disponible
def creaIndividuo(inversiones, capital):
    individuo = [0] * len(inversiones)
    
    while capitalInvertido(individuo) < capital:
        eleccion = randint(0, len(inversiones) - 1)
        individuo[eleccion] += 1
        
    return ajustaCapital(individuo)

# Crea un nuevo individuo cruzando otros dos (cuyas posiciones se 
# indican en el segundo parámetro)
def cruza(poblacion, posiciones):
    L = len(poblacion[0])
    
    # Toma los genes del primer progenitor y luego toma al azar
    # un segmento de entre 1 y L genes del segundo progenitor
    hijo = poblacion[posiciones[0]][:]
    inicio = randint(0, L - 1)
    fin = randint(inicio + 1, L)
    hijo[inicio:fin] = poblacion[posiciones[1]][inicio:fin]
    
    return ajustaCapital(hijo)

# Aplica mutaciones a un individuo según una tasa dada; garantiza
# que cumple las restricciones de capital e inversiones
def muta(individuo, tasaMutacion):
    mutado = []
    for i in range(len(individuo)):
        if random() > tasaMutacion:
            mutado.append(individuo[i])
        else:
            mutado.append(randint(0, inversiones[i].cantidad))
            
    return ajustaCapital(mutado)

# Hace evolucionar el sistema durante un número de generaciones
def evoluciona(poblacion, generaciones):
    
    # Ordena la población inicial por rendimiento producido
    poblacion.sort(key=lambda x: rendimiento(x))
    
    # Algunos valores útiles
    N = len(poblacion)
    tasaMutacion = 0.01
    
    # Genera una lista del tipo [0,1,1,2,2,2,3,3,3,3,...] para
    # representar las probabilidades de reproducirse de cada
    # individuo (el primero 1 posibilidad, el segundo 2, etc.)
    reproduccion = [x for x in range(N) for y in range(x + 1)]
    
    for i in range(generaciones):
        # Se generan N-1 nuevos individuos cruzando los existentes
        # (sin que se repitan los padres)
        padres = sample(reproduccion, 2)
        while padres[0] == padres[1]:
            padres = sample(reproduccion, 2)
        hijos = [cruza(poblacion, padres) for x in range(N - 1)]
        
        # Se aplican mutaciones con una cierta probabilidad
        hijos = [muta(x, tasaMutacion) for x in hijos]
        
        # Se añade el mejor individuo de la población anterior
        # (elitismo)
        hijos.append(poblacion[-1])
        poblacion = hijos
        
        # Se ordenan los individuos por rendimiento
        poblacion.sort(key=lambda x: rendimiento(x))
        
    # Devuelve el mejor individuo encontrado
    return poblacion[-1]



In [3]:
# Declara una tupla con nombres para representar cada inversión
Inversion = namedtuple('Inversion', 'precio cantidad rendim')

numInver = 100
maxPrecio = 1000
maxCant = 10
maxRend = 0.2

# Genera una lista de tuplas Inversion
inversiones = [Inversion(random() * maxPrecio, randint(1, maxCant),
               random() * maxRend) for i in range(numInver)]
print(inversiones)

capital = 50000
individuos = 20
generaciones = 1000

poblacion = [creaIndividuo(inversiones, capital)
             for i in range(individuos)]

# Nota: para simplificar el programa se accede a inversiones y
# capital de forma global (sólo se leen, no se modifican)

mejor = evoluciona(poblacion, generaciones)
print(mejor, capitalInvertido(mejor), rendimiento(mejor))

[Inversion(precio=303.27010563806624, cantidad=7, rendim=0.18604061214831968), Inversion(precio=589.6626408328877, cantidad=3, rendim=0.05757152787217943), Inversion(precio=274.91371490065865, cantidad=10, rendim=0.029696698744873996), Inversion(precio=140.22306728967826, cantidad=1, rendim=0.11004870030471126), Inversion(precio=84.35634918462809, cantidad=1, rendim=0.05261359899263321), Inversion(precio=296.43853641542427, cantidad=7, rendim=0.14547323835383025), Inversion(precio=424.15839616282847, cantidad=5, rendim=0.19671921591313932), Inversion(precio=550.6939894929424, cantidad=7, rendim=0.15217577672310406), Inversion(precio=790.8993238703576, cantidad=6, rendim=0.1594082403179885), Inversion(precio=467.02298081918445, cantidad=2, rendim=0.0075792248995307345), Inversion(precio=216.76562793180142, cantidad=6, rendim=0.12560068627146478), Inversion(precio=719.8224267927941, cantidad=10, rendim=0.19527933391888336), Inversion(precio=584.2857938783582, cantidad=6, rendim=0.0856888

*Codigo implementado con la libreria DEAP*

In [ ]:
# Implementacion con DEAP (corrige "deep" -> "deap")
from random import random, randint
from collections import namedtuple
import sys
import subprocess

# Anduve teniendo errores con la lib, por lo que hice este codigo
try:
    from deap import base, creator, tools, algorithms
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "deap"])
    from deap import base, creator, tools, algorithms


# Si no existen inversiones/capital, se crean valores por defecto
if "inversiones" not in globals() or "capital" not in globals():
    Inversion = namedtuple("Inversion", "precio cantidad rendim")
    numInver = 100
    maxPrecio = 1000
    maxCant = 10
    maxRend = 0.2
    inversiones = [
        Inversion(random() * maxPrecio, randint(1, maxCant), random() * maxRend)
        for _ in range(numInver)
    ]
    capital = 50000

MAX_GENES = [inv.cantidad for inv in inversiones]


def capital_invertido(individuo):
    return sum(cant * inv.precio for cant, inv in zip(individuo, inversiones))


def rendimiento_total(individuo):
    return sum(cant * inv.precio * inv.rendim for cant, inv in zip(individuo, inversiones))


def ajustar_capital(individuo):
    while capital_invertido(individuo) > capital:
        i = randint(0, len(individuo) - 1)
        if individuo[i] > 0:
            individuo[i] -= 1
    return individuo


# Evita redefinir clases de DEAP al re-ejecutar la celda
if not hasattr(creator, "FitnessMax"):
    creator.create("FitnessMax", base.Fitness, weights=(1.0,))
if not hasattr(creator, "Portfolio"):
    creator.create("Portfolio", list, fitness=creator.FitnessMax)


toolbox = base.Toolbox()


def init_portfolio():
    individuo = creator.Portfolio(randint(0, m) for m in MAX_GENES)
    return ajustar_capital(individuo)


def evaluar(individuo):
    ajustar_capital(individuo)
    return (rendimiento_total(individuo),)


def mutar(individuo, indpb=0.02):
    for i, m in enumerate(MAX_GENES):
        if random() < indpb:
            individuo[i] = randint(0, m)
    ajustar_capital(individuo)
    return (individuo,)


def cruzar(ind1, ind2):
    tools.cxTwoPoint(ind1, ind2)
    ajustar_capital(ind1)
    ajustar_capital(ind2)
    return ind1, ind2


toolbox.register("individual", init_portfolio)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)
toolbox.register("mate", cruzar)
toolbox.register("mutate", mutar, indpb=0.02)
toolbox.register("select", tools.selTournament, tournsize=3)
toolbox.register("evaluate", evaluar)


POP_SIZE = 20
NGEN = 1000
CXPB = 0.7
MUTPB = 0.2

poblacion_deap = toolbox.population(n=POP_SIZE)
hof = tools.HallOfFame(1)
estadisticas = tools.Statistics(lambda ind: ind.fitness.values[0])
estadisticas.register("max", max)
estadisticas.register("avg", lambda vals: sum(vals) / len(vals))

algorithms.eaSimple(
    poblacion_deap,
    toolbox,
    cxpb=CXPB,
    mutpb=MUTPB,
    ngen=NGEN,
    stats=estadisticas,
    halloffame=hof,
    verbose=False,
)

mejor_deap = hof[0]
print("Mejor individuo (DEAP):", mejor_deap)
print("Capital invertido:", capital_invertido(mejor_deap))
print("Rendimiento:", rendimiento_total(mejor_deap))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 565.6/565.6 kB 2.7 MB/s  0:00:0036m-:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [deap]1/2 [deap]



[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: pip install --upgrade pip


Mejor individuo (DEAP): [0, 0, 0, 0, 0, 2, 7, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 0, 0, 4, 0, 7, 0, 0, 1, 0, 7, 0, 0, 2, 5, 0, 0, 0, 6, 1, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 8, 8, 3, 3, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 7, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0]
Capital invertido: 49941.27693262978
Rendimiento: 9025.054478206535


Comparacion:

Diferencias en la programación de la función objetivo:
- Libro =  la función objetivo se hace "a mano" con rendimiento()
- DEAP =  también se calcula el rendimiento, pero se entrega en evaluar() como una lista.

Diferencias en la programación del fitness:
- libro = el fitness se usa de forma directa 
- DEAP = el fitness se define formalmente con creator.FitnessMax

Diferencias en la codificación de los operadores de selección, cruza y mutación:
- libro = selección, cruza y mutacionn estan programadas con funciones propias
- DEAp = se registran en toolbox y se usan operadores de la lib 
